- original point cloud
- fps sampled points
- random sampling comparison with fps
---------------------------------------------


In [11]:
import sys
sys.path.insert(0, "../")

import torch
import trimesh
import numpy as np
%matplotlib qt
import matplotlib.pyplot as plt

from barebones import fps

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [3]:
mesh = trimesh.load("3d-models/bunny.glb", force='mesh')

points, _ = trimesh.sample.sample_surface(mesh, 5000)

pts = torch.tensor(points, dtype=torch.float32)

In [ ]:
def random_sampling(pts, k):
    N = pts.shape[0]
    idx = torch.randperm(N)[:k]
    return idx, pts[idx]

K = 512

fps_idx, fps_pts = fps.fps(pts, K)

rand_idx, rand_pts = random_sampling(pts, K)

In [5]:
fps_mean = fps.mean_coverage(pts, fps_idx)
fps_max  = fps.max_coverage(pts, fps_idx)

rand_mean = fps.mean_coverage(pts, rand_idx)
rand_max  = fps.max_coverage(pts, rand_idx)

print("\n=== Coverage Metrics ===")
print(f"FPS    -> mean: {fps_mean:.6f}, max: {fps_max:.6f}")
print(f"Random -> mean: {rand_mean:.6f}, max: {rand_max:.6f}")


=== Coverage Metrics ===
FPS    -> mean: 12.142043, max: 24.370678
Random -> mean: 15.446178, max: 52.159691


In [ ]:
import matplotlib.pyplot as plt

methods = ["Random", "FPS"]
mean_vals = [rand_mean, fps_mean]
max_vals  = [rand_max, fps_max]

fig, ax = plt.subplots(1, 2, figsize=(10,4), dpi=150)

ax[0].bar(methods, mean_vals)
ax[0].set_title("Mean Coverage (↓ better)")
ax[0].set_ylabel("Distance")

# Max coverage
ax[1].bar(methods, max_vals)
ax[1].set_title("Worst-Case Gap (↓ better)")

plt.tight_layout()
plt.show()

In [6]:
def compute_heatmap(all_pts, sampled_pts):
    dists = torch.cdist(all_pts, sampled_pts)
    min_dist, _ = dists.min(dim=1)

    heat = (min_dist - min_dist.min()) / (min_dist.max() - min_dist.min())
    return heat.numpy()

In [7]:
fps_heat  = compute_heatmap(pts, fps_pts)
rand_heat = compute_heatmap(pts, rand_pts)

In [ ]:
def plot_cloud(ax, pts, heat, sampled, title):
    pts_np = pts.numpy()
    sampled_np = sampled.numpy()

    sc = ax.scatter(
        pts_np[:,0], pts_np[:,1], pts_np[:,2],
        c=heat,
        s=0.6,          
        cmap='inferno',        
        alpha=0.95,
        linewidths=0,
        depthshade=False      
    )

    ax.scatter(
        sampled_np[:,0], sampled_np[:,1], sampled_np[:,2],
        s=12,
        c='white',
        edgecolors='black',
        linewidth=0.4
    )

    ax.set_title(title, fontsize=11)
    ax.set_axis_off()

    ax.view_init(elev=20, azim=45)

    return sc

In [ ]:
fig = plt.figure(figsize=(10,5))

ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')
ax1.text2D(
    0.05, 0.95,
    "Large gaps here",
    transform=ax1.transAxes,
    color="red",
    fontsize=10
)

sc1 = plot_cloud(ax1, pts, rand_heat, rand_pts,
           "Random Sampling\n(red = missed regions)")
sc2 = plot_cloud(ax2, pts, fps_heat, fps_pts,
           "Farthest Point Sampling\nmore uniform coverage")

cbar = fig.colorbar(sc2, ax=[ax1, ax2], fraction=0.025, pad=0.04)
cbar.set_label("Coverage Error")

plt.tight_layout()
plt.show()

In [10]:
runs = 10
rand_means = []

for _ in range(runs):
    idx, _ = random_sampling(pts, K)
    m = fps.mean_coverage(pts, idx)
    rand_means.append(m)

rand_means = torch.tensor(rand_means)

print("\nRandom Stability:")
print(f"mean: {rand_means.mean():.6f}")
print(f"std:  {rand_means.std():.6f}")


Random Stability:
mean: 15.060872
std:  0.148971


In [ ]:
import numpy as np

methods = ["Random", "FPS"]
means = [rand_means.mean().item(), fps_mean]
stds  = [rand_means.std().item(), 0.0]

plt.figure(figsize=(5,4), dpi=150)
plt.bar(methods, means, yerr=stds, capsize=5)

plt.title("Sampling Stability (Mean Coverage)")
plt.ylabel("Distance (↓ better)")
plt.show()